# RetailMind AI Agent — StyleCraft Product Intelligence

**Company:** RetailMind Analytics  
**Client:** StyleCraft (D2C Fashion, 30 SKUs)  
**Architecture:** 2 Specialized Agents + RetailMindOrchestrator  
**LLM:** OpenAI gpt-4o-mini

## Agent Architecture
- **Agent 1 — InventoryAnalystAgent**: Inventory risk analysis (3 tools)
- **Agent 2 — BusinessIntelligenceAgent**: Pricing + reviews + category (3 tools)
- **RetailMindOrchestrator**: Runs both sequentially, passes accumulated context forward

## Tool-Calling Layer (6 Tools)
| Tool | Agent | Description |
|---|---|---|
| `search_products` | Inventory | Product search |
| `get_inventory_health` | Inventory | Per-product inventory status |
| `generate_restock_alert` | Inventory | All at-risk products sorted by urgency |
| `get_pricing_analysis` | BI | Gross margin + positioning |
| `get_review_insights` | BI | LLM-powered sentiment analysis |
| `get_category_performance` | BI | Category aggregated metrics |

## Cell 1: Imports & Environment Setup

In [ ]:
import os
import sys
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime
from dotenv import load_dotenv

# Add project root to path
sys.path.insert(0, os.path.abspath('.'))

from langchain_openai import ChatOpenAI
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain.memory import ConversationBufferMemory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage

load_dotenv()
print("✅ Environment loaded")
print(f"   OpenAI key set: {'Yes' if os.getenv('OPENAI_API_KEY') else 'No — set OPENAI_API_KEY in .env'}")

## Cell 2: Load & Preview Datasets

In [ ]:
from core.analytics import load_products, load_reviews

df_products = load_products('data/retailmind_products.csv')
df_reviews = load_reviews('data/retailmind_reviews.csv')

print(f"Products: {len(df_products)} rows × {len(df_products.columns)} columns")
print(f"Reviews : {len(df_reviews)} rows × {len(df_reviews.columns)} columns")
print("\nCategories:", df_products['category'].value_counts().to_dict())
df_products.head()

In [ ]:
df_reviews.head()

## Cell 3: Analytics Functions — Formula Verification

In [ ]:
from core.analytics import (
    compute_days_to_stockout, classify_inventory_status,
    compute_revenue_at_risk, compute_gross_margin,
    classify_price_positioning, get_category_avg_prices,
    run_full_pipeline
)

# Spot-check 5 products (verifies formulas for rubric: 25/30 products)
checks = [
    ('SC001', 45, 2.3, 1299, 480),
    ('SC004', 3, 5.8, 999, 360),
    ('SC010', 2, 4.7, 1599, 590),
    ('SC020', 1, 3.5, 2999, 1050),
    ('SC027', 88, 4.4, 499, 150),
]

print(f"{'ID':<8} {'Days Left':<12} {'Status':<10} {'Margin %':<10}")
print("-" * 42)
for pid, stock, sales, price, cost in checks:
    days = compute_days_to_stockout(stock, sales)
    status = classify_inventory_status(days)
    margin = compute_gross_margin(price, cost)
    print(f"{pid:<8} {days:<12.2f} {status:<10} {margin:<10.2f}%")

In [ ]:
# Run full pipeline and display KPIs
pipeline = run_full_pipeline(df_products, df_reviews)
df_enriched = pipeline['df_enriched']

print("📊 Catalog KPIs")
print(f"  Total SKUs          : {pipeline['total_skus']}")
print(f"  🔴 Critical stock   : {pipeline['critical_stock_count']}")
print(f"  🟡 Low stock        : {pipeline['low_stock_count']}")
print(f"  🟢 Healthy          : {pipeline['healthy_stock_count']}")
print(f"  Avg margin %        : {pipeline['avg_margin_pct']:.2f}%")
print(f"  Avg rating          : {pipeline['avg_rating']:.2f}/5")
print(f"  Rev. at risk (crit) : ₹{pipeline['total_revenue_at_risk_inr']:,.0f}")

## Cell 4: Load Tool Definitions (set session_state equivalent for notebook)

In [ ]:
# In notebook context, tools._get_dfs() falls back to loading from disk.
# We patch it here to use our in-memory DataFrames for faster execution.

import core.tools as _tools_module

_orig_get_dfs = _tools_module._get_dfs

def _patched_get_dfs():
    return df_products, df_reviews

_tools_module._get_dfs = _patched_get_dfs

from core.tools import (
    search_products, get_inventory_health, generate_restock_alert,
    get_pricing_analysis, get_review_insights, get_category_performance,
    ALL_TOOLS, INVENTORY_TOOLS, BI_TOOLS
)

print(f"✅ Loaded {len(ALL_TOOLS)} tools")
for t in ALL_TOOLS:
    print(f"   • {t.name}")

In [ ]:
# Direct tool invocation tests — verify all 6 tools return JSON
print("🔧 Tool Invocation Tests\n")

print("1. search_products('yoga leggings'):")
r = search_products.invoke({'query': 'yoga leggings'})
data = json.loads(r)
print(f"   → {data['count']} matches found")

print("\n2. get_inventory_health('SC004'):")
r = get_inventory_health.invoke({'product_id': 'SC004'})
data = json.loads(r)
print(f"   → {data['product_name']}: {data['days_to_stockout']} days ({data['status']})")

print("\n3. generate_restock_alert(threshold_days=7):")
r = generate_restock_alert.invoke({'threshold_days': 7})
data = json.loads(r)
print(f"   → {data['alert_count']} products at risk | Total rev at risk: ₹{data['total_revenue_at_risk_inr']:,.0f}")

print("\n4. get_pricing_analysis('SC001'):")
r = get_pricing_analysis.invoke({'product_id': 'SC001'})
data = json.loads(r)
print(f"   → Margin: {data['gross_margin_pct']:.1f}% | Positioning: {data['price_positioning']}")

print("\n5. get_category_performance('Accessories'):")
r = get_category_performance.invoke({'category': 'Accessories'})
data = json.loads(r)
print(f"   → {data['total_skus']} SKUs | Avg margin: {data['avg_margin_pct']:.1f}% | Avg rating: {data['avg_rating']}")

print("\n6. get_review_insights('SC001'):")
r = get_review_insights.invoke({'product_id': 'SC001'})
data = json.loads(r)
print(f"   → Rating: {data['avg_rating']} | Reviews: {data['total_reviews']}")
print(f"   → Summary: {data['sentiment_summary'][:100]}...")

## Cell 5: LLM Setup

In [ ]:
# Notebook LLM: Always OpenAI gpt-4o-mini
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.2,   # low randomness — consistent business analysis
    max_tokens=1024,   # sufficient detail without excessive output
    top_p=0.9,         # nucleus sampling — focused but natural responses
)
print(f"✅ LLM ready: {llm.model_name}")

## Cell 6: Router Pattern Demonstration (LLM-based, NOT keyword/regex)

In [ ]:
# Router Pattern: LLM classifies intent into 5 routes
# This is PURE LLM-based routing — no if/elif keyword matching

router_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an intent classifier for a retail product intelligence system.
Classify the user's query into EXACTLY one of these 5 categories:

  INVENTORY  — about stock levels, stockout risk, reorder needs, days of inventory left
  PRICING    — about margins, gross profit, pricing tiers, cost efficiency, markups
  REVIEWS    — about customer feedback, star ratings, complaints, product quality, sentiment
  CATALOG    — about finding products, category overviews, top performers, catalog discovery
  GENERAL    — greetings, meta questions, general retail knowledge unrelated to data

Respond with ONLY the category name (one word, uppercase). No explanation."""),
    ("human", "{query}")
])

router_chain = router_prompt | llm | StrOutputParser()

# Test with 10 varied queries (rubric: 8/10 correct classification)
test_queries = [
    ("Which products are about to run out of stock?",          "INVENTORY"),
    ("What is the profit margin on the Trench Coat?",          "PRICING"),
    ("What are customers saying about the Floral Summer Dress?", "REVIEWS"),
    ("Show me all Accessories products",                        "CATALOG"),
    ("Hello! What can you help me with?",                       "GENERAL"),
    ("How many days of inventory does SC020 have left?",        "INVENTORY"),
    ("Which product has the lowest gross margin?",              "PRICING"),
    ("Are customers happy with the Yoga Leggings?",             "REVIEWS"),
    ("What's the top revenue-generating product in Dresses?",   "CATALOG"),
    ("What does D2C mean?",                                     "GENERAL"),
]

correct = 0
print(f"{'Query':<55} {'Expected':<12} {'Predicted':<12} {'✓'}")
print("-" * 85)
for query, expected in test_queries:
    predicted = router_chain.invoke({"query": query}).strip().upper()
    match = "✅" if predicted == expected else "❌"
    if predicted == expected:
        correct += 1
    print(f"{query[:53]:<55} {expected:<12} {predicted:<12} {match}")

print(f"\nRouter Accuracy: {correct}/10 ({correct*10}%)")
print("Rubric requirement: 8/10 ✅" if correct >= 8 else "⚠️ Below 8/10 — review router prompt")

## Cell 7: Agent 1 — InventoryAnalystAgent

In [ ]:
from core.agent_factory import INVENTORY_AGENT_SYSTEM_PROMPT

memory_inv = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

prompt_inv = ChatPromptTemplate.from_messages([
    ("system", INVENTORY_AGENT_SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

agent_inv = create_tool_calling_agent(llm, INVENTORY_TOOLS, prompt_inv)
exec_inv = AgentExecutor(
    agent=agent_inv,
    tools=INVENTORY_TOOLS,
    memory=memory_inv,
    verbose=True,
    max_iterations=5,
    handle_parsing_errors=True,
)

print("✅ InventoryAnalystAgent ready")
print(f"   Tools: {[t.name for t in INVENTORY_TOOLS]}")

In [ ]:
# Test InventoryAnalystAgent
result = exec_inv.invoke({"input": "Which StyleCraft products are at critical risk of stockout right now?"})
print(result['output'])

## Cell 8: Agent 2 — BusinessIntelligenceAgent

In [ ]:
from core.agent_factory import BI_AGENT_SYSTEM_PROMPT

memory_bi = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

prompt_bi = ChatPromptTemplate.from_messages([
    ("system", BI_AGENT_SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

agent_bi = create_tool_calling_agent(llm, BI_TOOLS, prompt_bi)
exec_bi = AgentExecutor(
    agent=agent_bi,
    tools=BI_TOOLS,
    memory=memory_bi,
    verbose=True,
    max_iterations=5,
    handle_parsing_errors=True,
)

print("✅ BusinessIntelligenceAgent ready")
print(f"   Tools: {[t.name for t in BI_TOOLS]}")

In [ ]:
# Test BusinessIntelligenceAgent
result = exec_bi.invoke({"input": "Analyse the pricing health of the Dresses category and flag any low-margin products."})
print(result['output'])

## Cell 9: RetailMindOrchestrator

In [ ]:
AGENT_TASKS = {
    "inventory_analyst": (
        "Perform a comprehensive inventory analysis of StyleCraft's 30-product catalog. "
        "Use generate_restock_alert to identify all at-risk products. "
        "Then use get_inventory_health for the top 3 most critical items. "
        "Report: total critical count, product names, days remaining, revenue at risk, "
        "and prioritised reorder recommendations."
    ),
    "business_intelligence": (
        "Perform a business intelligence review of StyleCraft's catalog. "
        "1) Use get_category_performance for ALL 5 categories (Tops, Dresses, Bottoms, Outerwear, Accessories). "
        "2) Use get_pricing_analysis to identify the 3 products with lowest gross margin. "
        "3) Use get_review_insights for the 2 worst-rated products. "
        "Summarise: category health rankings, margin issues, and customer satisfaction findings."
    ),
}


class RetailMindOrchestrator:
    """
    Orchestrates InventoryAnalystAgent → BusinessIntelligenceAgent in sequence.
    Each agent receives the accumulated output of all prior agents as context.
    This ensures coherent analysis — BI agent can reference inventory findings.
    """

    def __init__(self, inventory_agent: AgentExecutor, bi_agent: AgentExecutor):
        self.named_agents = {
            "inventory_analyst": inventory_agent,
            "business_intelligence": bi_agent,
        }

    def run_full_analysis(self, verbose: bool = True) -> dict:
        """Run all agents sequentially, passing accumulated context forward."""
        results = {"timestamp": datetime.now().isoformat(), "errors": []}
        accumulated_context = ""

        for phase_num, (agent_name, agent_exec) in enumerate(
            self.named_agents.items(), start=1
        ):
            try:
                base_task = AGENT_TASKS[agent_name]
                task = (
                    f"PRIOR ANALYSIS:\n{accumulated_context}\n\nYOUR TASK:\n{base_task}"
                    if accumulated_context
                    else base_task
                )

                if verbose:
                    print(f"\n{'='*60}")
                    print(f"Phase {phase_num}: {agent_name.upper()}")
                    print(f"{'='*60}")

                output = agent_exec.invoke({"input": task})["output"]
                results[agent_name] = output
                accumulated_context += (
                    f"\n\n--- PHASE {phase_num} ({agent_name.upper()}) OUTPUT ---\n{output}"
                )

                if verbose:
                    print(f"\n✅ Phase {phase_num} complete.")

            except Exception as e:
                err_msg = f"Phase {phase_num} ({agent_name}): {e}"
                results["errors"].append(err_msg)
                results[agent_name] = None
                print(f"❌ ERROR: {err_msg}")

        return results


orchestrator = RetailMindOrchestrator(exec_inv, exec_bi)
print("✅ RetailMindOrchestrator ready")

In [ ]:
# Run full orchestrated analysis
analysis_results = orchestrator.run_full_analysis(verbose=True)
print(f"\n\n{'='*60}")
print("ORCHESTRATION COMPLETE")
print(f"Timestamp: {analysis_results['timestamp']}")
print(f"Errors: {analysis_results['errors'] or 'None'}")

## Cell 10: Daily Briefing Generation

In [ ]:
from core.agent_factory import generate_daily_briefing_text

briefing = generate_daily_briefing_text(df_products, df_reviews)
from IPython.display import Markdown, display
display(Markdown(briefing))

## Cell 11: 5 Plotly Visualisations

In [ ]:
# Chart 1: Price Comparison — Product vs Category Average
cat_avg = df_products.groupby('category')['price'].mean().reset_index()
cat_avg.columns = ['category', 'cat_avg']
df_plot = df_products.merge(cat_avg, on='category').sort_values(['category', 'price'])

fig1 = go.Figure()
fig1.add_trace(go.Bar(name='Product Price (₹)', x=df_plot['product_name'], y=df_plot['price'],
                      marker_color='#6366f1'))
fig1.add_trace(go.Bar(name='Category Avg (₹)', x=df_plot['product_name'], y=df_plot['cat_avg'],
                      marker_color='rgba(100,116,139,0.4)'))
fig1.update_layout(title='📊 Chart 1: Product Price vs Category Average',
                   barmode='group', xaxis_tickangle=-45, height=450)
fig1.show()

In [ ]:
# Chart 2: Gross Margin Health — Horizontal bar with threshold lines
from core.analytics import compute_gross_margin
df_enriched['gross_margin_pct'] = df_enriched.apply(
    lambda r: compute_gross_margin(r['price'], r['cost']), axis=1)
df_sorted = df_enriched.sort_values('gross_margin_pct', ascending=True)

colors = df_sorted['gross_margin_pct'].apply(
    lambda m: '#ef4444' if m < 20 else '#f59e0b' if m < 25 else '#22c55e'
)
fig2 = go.Figure(go.Bar(
    x=df_sorted['gross_margin_pct'], y=df_sorted['product_name'],
    orientation='h', marker_color=colors
))
fig2.add_vline(x=20, line_dash='dash', line_color='#ef4444', annotation_text='20% Alert')
fig2.add_vline(x=25, line_dash='dash', line_color='#f59e0b', annotation_text='25% Target')
fig2.update_layout(title='📊 Chart 2: Gross Margin Health', height=700, xaxis_title='Gross Margin %')
fig2.show()

In [ ]:
# Chart 3: Inventory Risk — Days to stockout by status
df_inv = df_enriched.sort_values('days_to_stockout').copy()
df_inv['days_capped'] = df_inv['days_to_stockout'].clip(upper=30)

fig3 = px.bar(df_inv, x='product_name', y='days_capped', color='status',
              color_discrete_map={'Critical': '#ef4444', 'Low': '#f59e0b', 'Healthy': '#22c55e'},
              title='📊 Chart 3: Inventory Risk — Days to Stockout', height=450)
fig3.add_hline(y=7, line_dash='dash', line_color='red', annotation_text='7-day Critical')
fig3.add_hline(y=14, line_dash='dot', line_color='orange', annotation_text='14-day Low')
fig3.update_layout(xaxis_tickangle=-45)
fig3.show()

In [ ]:
# Chart 4: Sales Velocity — avg_daily_sales per product
df_vel = df_enriched.sort_values('avg_daily_sales', ascending=False)
cat_avg_sales = df_vel['avg_daily_sales'].mean()

fig4 = px.bar(df_vel, x='product_name', y='avg_daily_sales', color='category',
              title='📊 Chart 4: Sales Velocity (Avg Daily Units Sold)', height=420)
fig4.add_hline(y=cat_avg_sales, line_dash='dash', line_color='#6366f1',
               annotation_text=f'Avg: {cat_avg_sales:.1f}')
fig4.update_layout(xaxis_tickangle=-45)
fig4.show()

In [ ]:
# Chart 5: Priority Matrix — Stockout risk vs Margin, bubble = revenue at risk
df_matrix = df_enriched.copy()
df_matrix['days_capped'] = df_matrix['days_to_stockout'].clip(upper=30)

fig5 = px.scatter(df_matrix, x='days_capped', y='gross_margin_pct',
                  size='revenue_at_risk', color='status',
                  color_discrete_map={'Critical': '#ef4444', 'Low': '#f59e0b', 'Healthy': '#22c55e'},
                  hover_name='product_name', size_max=60,
                  title='📊 Chart 5: Priority Matrix — Stockout Risk vs Margin', height=500,
                  labels={'days_capped': 'Days to Stockout (capped 30)',
                          'gross_margin_pct': 'Gross Margin %'})
fig5.add_vline(x=7, line_dash='dash', line_color='#ef4444')
fig5.add_hline(y=25, line_dash='dash', line_color='#f59e0b')
fig5.show()

## Cell 12: Executive Report (LLM-Generated)

In [ ]:
# Build executive summary from pipeline KPIs + orchestrator output
inv_output = analysis_results.get('inventory_analyst', 'N/A')
bi_output = analysis_results.get('business_intelligence', 'N/A')

exec_prompt = f"""You are a senior retail analyst. Based on the following analysis of StyleCraft's 
product catalog, write a 3-paragraph executive report for the Product Manager:

KPI SUMMARY:
- Total SKUs: {pipeline['total_skus']}
- Critical stock items: {pipeline['critical_stock_count']}
- Average gross margin: {pipeline['avg_margin_pct']:.1f}%
- Average rating: {pipeline['avg_rating']:.2f}/5
- Revenue at risk (critical items): ₹{pipeline['total_revenue_at_risk_inr']:,.0f}

INVENTORY ANALYSIS:
{inv_output[:800] if inv_output else 'Not available'}

BUSINESS INTELLIGENCE:
{bi_output[:800] if bi_output else 'Not available'}

Write:
1. Paragraph 1: Overall catalog health and most urgent issues
2. Paragraph 2: Top 3 recommended actions with business justification
3. Paragraph 3: Opportunities and positive indicators

Be specific, data-driven, and actionable."""

exec_report = llm.invoke([HumanMessage(content=exec_prompt)]).content

print("" + "=" * 70)
print("  RETAILMIND EXECUTIVE REPORT — StyleCraft Product Intelligence")
print("=" * 70)
display(Markdown(exec_report))

In [ ]:
# Export enriched recommendations to CSV
import os
os.makedirs('data', exist_ok=True)

export_cols = [
    'product_id', 'product_name', 'category', 'price', 'cost',
    'stock_quantity', 'avg_daily_sales', 'days_to_stockout', 'status',
    'gross_margin_pct', 'avg_rating', 'revenue_at_risk', 'reorder_level'
]
df_export = df_enriched[export_cols].sort_values('days_to_stockout').round(2)
df_export.to_csv('data/retailmind_recommendations.csv', index=False)
print(f"✅ Exported {len(df_export)} rows to data/retailmind_recommendations.csv")
df_export.head(10)